<h2 align='center'>AEROPULSE</h2>
<h3 align='center'>AIRLINE OPERATIONS, DELAY & RELIABILITY ANALYTICS</3>

<h5 align='center'>NOTEBOOK 03 - DATA QUALITY ASSESSMENT</h5>

**Objective:**

Assess the quality, completeness, consistency and validity

of the raw BTS airline Operational dataset before cleaning

**Importance:** 

Raw data will NOT be modified in this notebook.

Assessment Areas:
1. Dataset structure
2. Missingness
3. Duplicate records
4. Data types
5. Unique-value consistency
6. Range / domain check
7. Business-rule checks
8. Data-quality summary
9. Cleaning recommendations

### Import Path

In [1]:
from pathlib import Path
import zipfile
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [2]:
PROJECT_PATH = Path.cwd().parent


RAW_PATH = PROJECT_PATH / "01_Raw_Data"
DOCUMENTATION_PATH = PROJECT_PATH / "02_Documentation"

print("Project_Path", PROJECT_PATH.resolve())
print("Raw_Path", RAW_PATH.resolve)
print("Documentation_Path", DOCUMENTATION_PATH.resolve())

Project_Path D:\DA_32 PYTHON\Aeropulse_Airline_Analytics\notebooks
Raw_Path <bound method Path.resolve of WindowsPath('d:/DA_32 PYTHON/Aeropulse_Airline_Analytics/notebooks/01_Raw_Data')>
Documentation_Path D:\DA_32 PYTHON\Aeropulse_Airline_Analytics\notebooks\02_Documentation


### Load Raw Data

In [3]:
csv_files = list(
    (RAW_PATH / "2024").rglob("*.csv")
)

if not csv_files:
    raise FileNotFoundError(
        "No CSV file found inside 01_Raw_Data/2024"
    )

RAW_FILE = csv_files[0]

df = pd.read_csv(
    RAW_FILE,
    low_memory=False
)

print("File:", RAW_FILE.name)
print("shape:", df.shape)

File: On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2024_1.csv
shape: (547271, 110)


### Data Quality Assessent Baseline

In [4]:
baseline = pd.DataFrame({
    "Metric": [
        "Total Rows",
        "Total Columns",
        "Total Cells",
        "Duplicate Rows",
        "Completely Empty Columns",
        "Columns with Missing values",
        "Columns without Missing values"
    ],
    "value":[
        len(df),
        len(df.columns),
        df.shape[0] * df.shape[1],
        df.duplicated().sum(),
        df.isna().all().sum(),
        (df.isna().any()).sum(),
        (~df.isna().any()).sum()
    ]
})

display(baseline)

,Metric,value
0,Total Rows,547271
1,Total Columns,110
2,Total Cells,60199810
3,Duplicate Rows,0
4,Completely Empty Columns,17
5,Columns with Missing values,71
6,Columns without Missing values,39


### Structural Assessment

In [5]:
print("========== DATASET STRUCTURE ==========")

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nMemory Usage:")
print(
    f"{df.memory_usage(deep=True).sum() / (1024**2):.2f} MB"
)

========== DATASET STRUCTURE ==========
Rows: 547271
Columns: 110

Memory Usage:
894.03 MB


### Column-Level Data Inventory

In [6]:
data_inventory = pd.DataFrame({
    "Column": df.columns,
    "Data_Type": df.dtypes.astype(str),
    "Non_Null_Count": df.notna().sum().values,
    "Null_Count": df.isna().sum().values,
    "Null_Percentage": (
        df.isna().mean().values * 100
    ). round(2),
    "Unique_Values": df.nunique(
        dropna=True
    ).values
})

display(data_inventory)

,Column,Data_Type,Non_Null_Count,Null_Count,Null_Percentage,Unique_Values
Year,Year,int64,547271,0,0.00,1
Quarter,Quarter,int64,547271,0,0.00,1
Month,Month,int64,547271,0,0.00,1
DayofMonth,DayofMonth,int64,547271,0,0.00,31
DayOfWeek,DayOfWeek,int64,547271,0,0.00,7
...,...,...,...,...,...,...
Div5TotalGTime,Div5TotalGTime,float64,0,547271,100.00,0
Div5LongestGTime,Div5LongestGTime,float64,0,547271,100.00,0
Div5WheelsOff,Div5WheelsOff,float64,0,547271,100.00,0
Div5TailNum,Div5TailNum,float64,0,547271,100.00,0


### Missingness Profile

In [7]:
missing_profile = (
    data_inventory[
        data_inventory["Null_Count"] > 0
    ]
    .sort_values(
        "Null_Percentage",
        ascending=False
    )
    .reset_index(drop=True)
)

display(missing_profile)

,Column,Data_Type,Non_Null_Count,Null_Count,Null_Percentage,Unique_Values
0,Div5WheelsOff,float64,0,547271,100.00,0
1,Unnamed: 109,float64,0,547271,100.00,0
2,Div4LongestGTime,float64,0,547271,100.00,0
3,Div4TotalGTime,float64,0,547271,100.00,0
4,Div4WheelsOn,float64,0,547271,100.00,0
5,Div4AirportSeqID,float64,0,547271,100.00,0
6,Div4AirportID,float64,0,547271,100.00,0
7,Div3AirportID,float64,1,547270,100.00,1
8,Div3AirportSeqID,float64,1,547270,100.00,1
9,Div3WheelsOn,float64,1,547270,100.00,1


### Missingness Classification

In [8]:
missing_profile ["Missingness_Category"] = np.select(
    [
        missing_profile["Null_Percentage"] ==0,
        missing_profile["Null_Percentage"] < 5,
        missing_profile["Null_Percentage"] < 30,
        missing_profile["Null_Percentage"] < 70,
        missing_profile["Null_Percentage"] < 100,
        missing_profile["Null_Percentage"] == 100
    ],
    [
        "Complete",
        "Low",
        "Moderate",
        "High",
        "Very High",
        "Completely Missing"
    ],
    default="Review"
)

display(missing_profile)

,Column,Data_Type,Non_Null_Count,Null_Count,Null_Percentage,Unique_Values,Missingness_Category
0,Div5WheelsOff,float64,0,547271,100.00,0,Completely Missing
1,Unnamed: 109,float64,0,547271,100.00,0,Completely Missing
2,Div4LongestGTime,float64,0,547271,100.00,0,Completely Missing
3,Div4TotalGTime,float64,0,547271,100.00,0,Completely Missing
4,Div4WheelsOn,float64,0,547271,100.00,0,Completely Missing
5,Div4AirportSeqID,float64,0,547271,100.00,0,Completely Missing
6,Div4AirportID,float64,0,547271,100.00,0,Completely Missing
7,Div3AirportID,float64,1,547270,100.00,1,Completely Missing
8,Div3AirportSeqID,float64,1,547270,100.00,1,Completely Missing
9,Div3WheelsOn,float64,1,547270,100.00,1,Completely Missing


### Completely Empty Column

In [9]:
empty_columns = df.columns[
    df.isna().all()
].tolist()

print("Completely Empty Columns:", len(empty_columns))

for column in empty_columns:
    print(column)

Completely Empty Columns: 17
Div4Airport
Div4AirportID
Div4AirportSeqID
Div4WheelsOn
Div4TotalGTime
Div4LongestGTime
Div4WheelsOff
Div4TailNum
Div5Airport
Div5AirportID
Div5AirportSeqID
Div5WheelsOn
Div5TotalGTime
Div5LongestGTime
Div5WheelsOff
Div5TailNum
Unnamed: 109


### Duplicate Assessment

In [10]:
duplicate_count = df.duplicated().sum()

print("Complete Duplicate Rows:", duplicate_count)

Complete Duplicate Rows: 0


### Data-Type Quality Assessment

In [11]:
object_columns = df.select_dtypes(
    include="object"
).columns.tolist()

print("Object Columns:", len(object_columns))

for column in object_columns:
    print(column)

Object Columns: 21
FlightDate
Reporting_Airline
IATA_CODE_Reporting_Airline
Tail_Number
Origin
OriginCityName
OriginState
OriginStateName
Dest
DestCityName
DestState
DestStateName
DepTimeBlk
ArrTimeBlk
CancellationCode
Div1Airport
Div1TailNum
Div2Airport
Div2TailNum
Div3Airport
Div3TailNum


C:\Users\bhaje\AppData\Local\Temp\ipykernel_10280\1878942163.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_columns = df.select_dtypes(


### Data Quality

In [12]:
flight_date_test = pd.to_datetime(
    df["FlightDate"],
    errors='coerce'
)

invalid_flight_dates = (
    flight_date_test.isna() &
    df["FlightDate"].notna()
).sum()

print(
    "Invalid / Unparseable FlightDate:",
    invalid_flight_dates
)

print(
   "Minimum Flight Date:",
    flight_date_test.min()
)

print(
    "Maximum Flight Date:",
    flight_date_test.max()
)

Invalid / Unparseable FlightDate: 0
Minimum Flight Date: 2024-01-01 00:00:00
Maximum Flight Date: 2024-01-31 00:00:00


### Categorical Consistency

In [13]:
categorical_columns = [
    "Reporting_Airline",
    "Origin",
    "Dest",
    "CancellationCode",
    "DepTimeBlk",
    "ArrTimeBlk"
]

categorical_columns = [
    c for c in categorical_columns
    if c in df.columns
]

for column in categorical_columns:
    print(f"\n========= {column} =========")
    print(df[column].value_counts(dropna=False).head(20))


========= Reporting_Airline =========
Reporting_Airline
WN    115389
AA     77346
DL     74384
UA     58855
OO     56814
YX     22914
MQ     20750
NK     20415
B6     19580
AS     17775
9E     16972
OH     16526
F9     14379
G4      8596
HA      6576
Name: count, dtype: int64

========= Origin =========
Origin
ATL    26315
DFW    23570
DEN    23361
ORD    20321
CLT    16378
PHX    15378
LAX    15228
LAS    14942
MCO    14296
LGA    12700
SEA    11751
DCA    11486
MIA    10214
EWR    10176
BOS    10134
SFO    10133
DTW     9473
JFK     9471
SLC     9459
MSP     9438
Name: count, dtype: int64

========= Dest =========
Dest
ATL    26294
DFW    23588
DEN    23357
ORD    20327
CLT    16382
PHX    15367
LAX    15220
LAS    14943
MCO    14304
LGA    12718
SEA    11757
DCA    11487
MIA    10210
EWR    10179
SFO    10128
BOS    10126
DTW     9470
JFK     9467
SLC     9455
MSP     9431
Name: count, dtype: int64

========= CancellationCode =========
CancellationCode
NaN    526882
B       12085
A

### Binary Field Assessment

In [14]:
binary_columns = [
    "Cancelled",
    "Diverted",
    "DepDel15",
    "ArrDel15"
]

for column in binary_columns:

    if column in df.columns:

        print(f"\n========= {column} =========")

        print(
            df[column]
            .value_counts(dropna=False)
        )


========= Cancelled =========
Cancelled
0.00    526882
1.00     20389
Name: count, dtype: int64

========= Diverted =========
Diverted
0.00    545759
1.00      1512
Name: count, dtype: int64

========= DepDel15 =========
DepDel15
0.00    405154
1.00    122259
NaN      19858
Name: count, dtype: int64

========= ArrDel15 =========
ArrDel15
0.00    398960
1.00    126410
NaN      21901
Name: count, dtype: int64


### Range Assessment

In [15]:
range_checks = {
    "Quarter outside 1-4":
        (~df["Quarter"].between(1, 4)).sum(),

    "Month Outside 1-12":
        (~df["Month"].between(1,12)).sum(),

    "DayOfWeek Outside 1-7":
        (~df["DayOfWeek"].between(1, 7)).sum(),

    "Distance <= 0":
        (df["Distance"] < 0).sum(),

    "Flights <= 0":
        (df["Flights"] <= 0).sum()
}

range_quality = pd.DataFrame({
    "validation_Rule": range_checks.keys(),
    "violation_Count": range_checks.values()
})

display(range_checks)

{'Quarter outside 1-4': np.int64(0),
 'Month Outside 1-12': np.int64(0),
 'DayOfWeek Outside 1-7': np.int64(0),
 'Distance <= 0': np.int64(0),
 'Flights <= 0': np.int64(0)}

### Cancellation Business Rule

In [16]:
cancelled_without_code = (
    (df["Cancelled"] == 1) &
    df["CancellationCode"].isna()
).sum()

notcancelled_with_code = (
    (df["Cancelled"] == 0) &
    df["CancellationCode"].notna()
).sum()

cancellation_quality = pd.DataFrame({
    "Rule":[
        "Cancelled flight without cancellation code",
        "Non-cancelled flight with cancellation code"
    ]
})

display(cancellation_quality)

,Rule
0,Cancelled flight without cancellation code
1,Non-cancelled flight with cancellation code


### Delay Indicator Business Rules

In [17]:
dep_delay_issue = (
    (df["DepDel15"] == 1) &
    (df["DepDelayMinutes"] < 15)
).sum()

arr_delay_issue = (
    (df["ArrDel15"] == 1) &
    (df["ArrDelayMinutes"] < 15)
).sum()

delay_quality = pd.DataFrame({
    "Issue_Type": [
        "DepDel15 = 1 but DepDelayMinutes < 15",
        "ArrDel15 = 1 but ArrDelayMinutes < 15"
    ],
    "Issue_Count": [
        dep_delay_issue,
        arr_delay_issue
    ]
})

display(delay_quality)

,Issue_Type,Issue_Count
0,DepDel15 = 1 but DepDelayMinutes < 15,0
1,ArrDel15 = 1 but ArrDelayMinutes < 15,0


In [18]:
negative_delay_profile = pd.DataFrame({
    "Metric": [
        "Negative Departure Delay",
        "Negative Arrival Delay"
    ],
    "Count":[
        (df["DepDelay"] < 0).sum(),
        (df["ArrDelay"] < 0).sum()
    ]
})

display(negative_delay_profile)

,Metric,Count
0,Negative Departure Delay,292317
1,Negative Arrival Delay,306999


### Diversion-Specific Missingness

In [19]:
diversion_quality = pd.DataFrame({
    "Metric": [
        "Total Diverted Flight",
        "Diverted Flights with ArrDelay NULL",
        "Diverted Flights with ArrDelay populated"
    ],
    "count": [
        (df["Diverted"] == 1).sum(),

        (
            (df["Diverted"] == 1) &
            df["ArrDelay"].isna()
        ).sum(),
        (
            (df["Diverted"] == 1) &
            df["ArrDelay"].notna()
        ).sum()
    ]
})

display(diversion_quality)

,Metric,count
0,Total Diverted Flight,1512
1,Diverted Flights with ArrDelay NULL,1512
2,Diverted Flights with ArrDelay populated,0


------------------------------------------------------------------------------------------------------